<a href="https://colab.research.google.com/github/HSA2005-cloud/BIS-Lab/blob/main/4.%20Cuckoo%20Search%20Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import numpy as np
import random
import math

n = 25
pa = 0.25
MAX_ITER = 100

alpha = 0.01
beta = 1.5

battery_capacity = 100
max_charge = 20
max_discharge = 20

initial_soc = 50
min_soc = 10
max_soc = 100

charge_efficiency = 0.95
discharge_efficiency = 0.95

solar = np.array([
    0, 0, 0, 0, 2, 5,
    10, 20, 35, 45, 55, 60,
    65, 60, 50, 35, 20, 10,
    3, 0, 0, 0, 0, 0
])

wind = np.array([
    15, 18, 20, 17, 15, 12,
    10, 12, 15, 18, 20, 22,
    20, 18, 15, 17, 20, 22,
    25, 23, 20, 18, 17, 16
])

demand = np.array([
    30, 28, 25, 24, 25, 30,
    40, 50, 55, 60, 65, 70,
    75, 70, 65, 60, 55, 60,
    70, 75, 70, 60, 45, 35
])

grid_price = np.array([
    5, 5, 5, 5, 5, 5,
    6, 6, 7, 7, 8, 8,
    8, 8, 7, 7, 7, 8,
    9, 9, 9, 7, 6, 5
])

def fitness(schedule):

    soc = initial_soc

    total_cost = 0
    total_grid = 0
    total_renewable = 0
    total_curtailed = 0

    for t in range(24):

        renewable = solar[t] + wind[t]

        power = schedule[t]

        # Battery operation
        if power < 0:

            # Charging
            charge = min(
                abs(power),
                max_charge,
                (max_soc - soc) / charge_efficiency
            )

            soc += charge * charge_efficiency

            actual_power = -charge

        else:

            # Discharging
            discharge = min(
                power,
                max_discharge,
                (soc - min_soc) * discharge_efficiency
            )

            soc -= discharge / discharge_efficiency

            actual_power = discharge

        # Calculate grid requirement
        grid = max(
            0,
            demand[t] - renewable
            - actual_power
        )

        # Calculate excess renewable energy
        excess = max(
            0,
            renewable + actual_power
            - demand[t]
        )

        # Renewable energy utilized
        utilized = renewable - min(renewable, excess)

        # Operating cost
        cost = (
            grid * grid_price[t]
            + abs(actual_power) * 0.05
            + excess * 2
        )

        total_cost += cost

        total_grid += grid
        total_renewable += utilized
        total_curtailed += excess

    return (
        total_cost,
        total_grid,
        total_renewable,
        total_curtailed
    )

def levy_flight():

    sigma = (
        math.gamma(1 + beta)
        * math.sin(math.pi * beta / 2)
        /
        (
            math.gamma((1 + beta) / 2)
            * beta
            * 2 ** ((beta - 1) / 2)
        )
    ) ** (1 / beta)

    u = np.random.normal(0, sigma, 24)
    v = np.random.normal(0, 1, 24)

    step = u / (np.abs(v) ** (1 / beta))

    return step

nests = np.random.uniform(
    -max_charge,
    max_discharge,
    (n, 24)
)

fitness_values = [
    fitness(nest)[0] for nest in nests
]

best_index = np.argmin(fitness_values)

best_nest = nests[best_index].copy()
best_fitness = fitness_values[best_index]


for iteration in range(MAX_ITER):

    # Generate new solutions using Levy flights
    for i in range(n):

        new_nest = (
            nests[i]
            + alpha * levy_flight()
            * (nests[i] - best_nest)
        )

        new_nest = np.clip(
            new_nest,
            -max_charge,
            max_discharge
        )

        new_fitness = fitness(new_nest)[0]

        # Replace nest if new solution is better
        if new_fitness < fitness_values[i]:

            nests[i] = new_nest
            fitness_values[i] = new_fitness

    # Abandon worst nests
    abandon_count = int(pa * n)

    worst_indices = np.argsort(
        fitness_values
    )[-abandon_count:]

    for i in worst_indices:

        nests[i] = np.random.uniform(
            -max_charge,
            max_discharge,
            24
        )

        fitness_values[i] = fitness(nests[i])[0]

    # Update best solution
    best_index = np.argmin(fitness_values)

    if fitness_values[best_index] < best_fitness:

        best_fitness = fitness_values[best_index]
        best_nest = nests[best_index].copy()

total_cost, total_grid, total_renewable, total_curtailed = fitness(best_nest)

total_demand = np.sum(demand)
total_available = np.sum(solar + wind)

renewable_utilization = (
    (total_available - total_curtailed)
    / total_available
) * 100

grid_dependency = (
    total_grid / total_demand
) * 100


print("\n========== CS PARAMETERS ==========")

print("Number of Nests:", n)
print("Discovery Probability (pa):", pa)
print("Maximum Iterations:", MAX_ITER)
print("Levy Step Size (alpha):", alpha)
print("Levy Distribution Parameter (beta):", beta)

print("\n========== BATTERY PARAMETERS ==========")

print("Battery Capacity:", battery_capacity, "kWh")
print("Maximum Charging Power:", max_charge, "kW")
print("Maximum Discharging Power:", max_discharge, "kW")
print("Initial SOC:", initial_soc, "kWh")
print("Minimum SOC:", min_soc, "kWh")
print("Maximum SOC:", max_soc, "kWh")
print("Charging Efficiency:", charge_efficiency)
print("Discharging Efficiency:", discharge_efficiency)

print("\n========== OPTIMIZATION RESULTS ==========")

print("Minimum Operating Cost:",
      round(total_cost, 2))

print("Grid Electricity Consumption:",
      round(total_grid, 2), "kWh")

print("Renewable Energy Utilized:",
      round(total_renewable, 2), "kWh")

print("Renewable Energy Utilization:",
      round(renewable_utilization, 2), "%")

print("Grid Dependency:",
      round(grid_dependency, 2), "%")

print("Curtailed Renewable Energy:",
      round(total_curtailed, 2), "kWh")

print("\n========== OPTIMAL BATTERY SCHEDULE ==========")

for hour in range(24):

    power = best_nest[hour]

    if power < 0:
        print(
            "Hour", hour + 1,
            ": Charge",
            round(abs(power), 2), "kW"
        )

    else:
        print(
            "Hour", hour + 1,
            ": Discharge",
            round(power, 2), "kW"
        )

print("\nOptimization Completed!")



========== CS PARAMETERS ==========
Number of Nests: 25
Discovery Probability (pa): 0.25
Maximum Iterations: 100
Levy Step Size (alpha): 0.01
Levy Distribution Parameter (beta): 1.5

========== BATTERY PARAMETERS ==========
Battery Capacity: 100 kWh
Maximum Charging Power: 20 kW
Maximum Discharging Power: 20 kW
Initial SOC: 50 kWh
Minimum SOC: 10 kWh
Maximum SOC: 100 kWh
Charging Efficiency: 0.95
Discharging Efficiency: 0.95

========== OPTIMIZATION RESULTS ==========
Minimum Operating Cost: 2468.84
Grid Electricity Consumption: 352.13 kWh
Renewable Energy Utilized: 857.34 kWh
Renewable Energy Utilization: 95.26 %
Grid Dependency: 28.35 %
Curtailed Renewable Energy: 42.66 kWh

========== OPTIMAL BATTERY SCHEDULE ==========
Hour 1 : Charge 4.57 kW
Hour 2 : Charge 1.49 kW
Hour 3 : Charge 8.58 kW
Hour 4 : Charge 16.48 kW
Hour 5 : Charge 19.7 kW
Hour 6 : Charge 17.13 kW
Hour 7 : Charge 8.41 kW
Hour 8 : Charge 17.73 kW
Hour 9 : Charge 3.62 kW
Hour 10 : Charge 11.36 kW
Hour 11 : Discharge 3